In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import h5py
from pathlib import Path

In [2]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
transport_dir = base_dir / "Updated Code" / "transport narrowed param ranges"
mat_path = transport_dir / "all_trial_bkt_data.mat"

In [3]:
def load_all_trial_bkt_data(mat_path):
    param_sets = []
    with h5py.File(mat_path, 'r') as f:
        weights = f['weights'][()].T  # (num_param_sets, 3)
        atbd = f['all_trial_bkt_data']  # (1, N) object refs
        for i in range(atbd.shape[1]):
            ps_grp = f[atbd[0, i]]
            p0 = float(ps_grp['p0'][()].flat[0])
            td = ps_grp['trial_data']  # (1, 1000) object refs
            trial_data = []
            for j in range(td.shape[1]):
                arr = f[td[0, j]][()]
                if arr.dtype == np.float64:
                    trial_data.append(arr.flatten())
                else:
                    trial_data.append(np.array([], dtype=np.float64))
            param_sets.append({'p0': p0, 'trial_data': trial_data})
    return param_sets, weights


def recover_theta(w):
    disc = 1*w[0] + 2.1*(1 - w[0])       # [1, 2.1]
    b    = 0.01e-3*w[1] + 0.3e-3*(1 - w[1])  # [0.01e-3, 0.3e-3]
    lr   = 0.3*w[2] + 0.5*(1 - w[2])     # [0.3, 0.5]
    return disc, lr, b


param_sets, weights = load_all_trial_bkt_data(mat_path)
print(f"Loaded {len(param_sets)} param sets, {len(param_sets[0]['trial_data'])} trials each")

Loaded 100 param sets, 1000 trials each


In [4]:
# Time points and plot settings — adjust as needed after first inspection
t_vals  = np.logspace(5, 6, 5)   # [1e5, ~1.8e5, ~3.2e5, ~5.6e5, 1e6] seconds
colors  = plt.cm.plasma(np.linspace(0.1, 0.9, len(t_vals)))
N_BINS  = 60
print('Time points (s):', [f'{t:.2e}' for t in t_vals])

Time points (s): ['1.00e+05', '1.78e+05', '3.16e+05', '5.62e+05', '1.00e+06']


In [ ]:
N_PARAM_SETS = 20
pdf_path = transport_dir / '[per t] distrib of percentiles reached by time t.pdf'

with PdfPages(pdf_path) as pdf:
    for ps_idx in range(N_PARAM_SETS):
        ps = param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])

        fig, ax = plt.subplots(figsize=(8, 5))

        for t, color in zip(t_vals, colors):
            s_vals = []
            for bkt in ps['trial_data']:
                if len(bkt) == 0:
                    s_vals.append(1.0)               # disconnected: all particles remain
                else:
                    s_vals.append(np.mean(bkt > t))  # fraction with BKT > t

            counts, edges = np.histogram(s_vals, bins=N_BINS, range=(0.0, 1.0), density=True)
            centres = (edges[:-1] + edges[1:]) / 2
            ax.plot(centres, counts, color=color, label=f't = {t:.2e} s')

        ax.set_xlabel('fraction of particles remaining in network')
        ax.set_ylabel('probability density')
        ax.set_xlim(0, 1)
        ax.set_title(
            f'Param set {ps_idx+1}  disc={disc:.2f}, lr={lr:.2f}, '
            f'b={b:.2e}, p0={ps["p0"]:.2f}')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}')

Saved /Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP/outputs/bkt_percentile_distributions.pdf
